# AKANE E12：辞書の説明文の頭の語の意味を、Gemmaにたくさん選ばせる

このノートは **材料を取るだけ** です（比べは結果のファイルを読んで別に行います）。計画：plans/E12_gloss_head_sense.md。

- 問題は WordNet 3.0 の名詞の説明文 42,257 問（頭の語 34,402、「〜の」の後ろの語 7,855）。
- 聞き方は E8 の方法D と同じ文です。候補はくじで並べ替えた順とその逆順の2通りで聞きます。
- 正解（人が付けた意味）はこのノートに入っていません。手で書いた語の一覧もありません。

## 手順
1. 「ランタイム」→「ランタイムのタイプを変更」→ **L4 GPU**。
2. 鍵マークの `HF_TOKEN` にアクセスを許可（いつもと同じ）。
3. 「ランタイム」→「すべてのセルを実行」。ドライブの許可を聞かれたら許可。
4. 終わると、ドライブの `AKANE_アーキテクチャ/E12_results/E12_results.json.gz` ができます。
5. 時間の見込み：40〜70分。途中で切れても、もう一度「すべてのセルを実行」すれば済んだ所（1万問ごとに保存）は飛ばします。

In [ ]:
import os, json, time, gzip, base64
import numpy as np
from google.colab import drive, userdata
drive.mount("/content/drive")
BASE = "/content/drive/MyDrive/AKANE_アーキテクチャ"
OUT = f"{BASE}/E12_results"
os.makedirs(OUT, exist_ok=True)
GEMMA_ID = "google/gemma-2-2b-it"   # E6〜E11と同じ
EMB_ID = "sentence-transformers/all-mpnet-base-v2"   # E8の方法Cと同じ

In [ ]:
# 中身（analysis/e12/e12_core.py と同じ。手書きの語の一覧はありません）
"""E12 の中身（Colabと手元の両方で使う）。聞き方は E8 の方法D（e8_core.choice_prompt）と同じ文。
候補は WordNet の順だと最初の意味（よく使う順）が先頭に来て答えが漏れるので、くじで並べ替えた順とその逆順の2通りで聞いて平均する（E11-1と同じ）。
手書きの語の一覧はない。"""
import numpy as np

MAX_SENSES = 8


def choice_prompt(w, d, g, cands):
    opts = "\n".join(f"{i + 1}. {c}" for i, c in enumerate(cands))
    return (f"Dictionary definition of \"{w}\": {d}\n\nIn this definition, which meaning of the word \"{g}\" is used?\n"
            f"{opts}\n\nAnswer with the number only.")


def jobs(items, heads, seed=0, shuffle=True):
    """items: [[id, kind, word, defn, head]], heads: {head: [説明文]}。shuffle=True なら1問2通り、False なら WordNet の順で1通り。"""
    rng = np.random.default_rng(seed)
    out = []
    for _id, _k, w, d, g in items:
        defs = heads[g][:MAX_SENSES]
        if shuffle:
            o = rng.permutation(len(defs)).tolist()
            orders = [o, o[::-1]]
        else:
            orders = [list(range(len(defs)))]
        for od in orders:
            out.append([choice_prompt(w, d, g, [defs[i] for i in od]), od])
    return out


def merge(jb, probs, per):
    """per 通りずつ、見せた順の値を元の順に戻して平均する。"""
    out = []
    for i in range(0, len(jb), per):
        acc = None
        for j in range(i, i + per):
            order, p = jb[j][1], probs[j]
            v = np.zeros(len(order))
            for pos, orig in enumerate(order):
                v[orig] = p[pos]
            acc = v if acc is None else acc + v
        out.append((acc / per).round(4).tolist())
    return out

In [ ]:
DATA = json.loads(gzip.decompress(base64.b64decode('...')))
ITEMS, HEADS, G0 = DATA['items'], DATA['heads'], DATA['g0']
print(len(ITEMS), '問  頭の語の種類', len(HEADS), '  WordNetの順で聞く分', len(G0))

## 1. 説明文どうしの近さ（方法C、GPU、2〜4分）

In [ ]:
import torch
from sentence_transformers import SentenceTransformer
ck = f"{OUT}/ck_sim.json"
if os.path.exists(ck):
    SIM = json.load(open(ck))
else:
    emb = SentenceTransformer(EMB_ID, device="cuda")
    texts = sorted({d for x in ITEMS for d in [x[3]]} | {d for v in HEADS.values() for d in v})
    V = emb.encode(texts, batch_size=256, normalize_embeddings=True, show_progress_bar=True)
    pos = {t: i for i, t in enumerate(texts)}
    SIM = [[round(float(V[pos[x[3]]] @ V[pos[d]]), 4) for d in HEADS[x[4]]] for x in ITEMS]
    json.dump(SIM, open(ck, "w"))
    del emb; torch.cuda.empty_cache()
print("近さ 済み", len(SIM))

## 2. Gemma に意味を選ばせる（方法D、GPU、40〜60分。1万問ごとにドライブへ保存）

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
tok = AutoTokenizer.from_pretrained(GEMMA_ID, token=userdata.get("HF_TOKEN"), padding_side="left")
gm = AutoModelForCausalLM.from_pretrained(GEMMA_ID, token=userdata.get("HF_TOKEN"), torch_dtype=torch.float16, device_map="cuda").eval()
NUM = [tok.encode(str(i), add_special_tokens=False)[0] for i in range(1, MAX_SENSES + 1)]
assert len(set(NUM)) == MAX_SENSES

def ask(jb, ck, B=32):
    if os.path.exists(ck):
        c = json.load(open(ck))
        if c.get("n") == len(jb) and c.get("head") == jb[0][0][:200]:
            print("ドライブから読み込み", ck); return c["out"]
    out = []; t = time.time()
    order = sorted(range(len(jb)), key=lambda i: len(jb[i][0]))   # 長さをそろえて速く
    res = [None] * len(jb)
    for s in range(0, len(order), B):
        idx = order[s:s + B]
        ps = [tok.apply_chat_template([{"role": "user", "content": jb[i][0]}], tokenize=False, add_generation_prompt=True) for i in idx]
        enc = tok(ps, return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
        with torch.no_grad():
            last = gm(**enc, logits_to_keep=1).logits[:, -1, :].float()
        for row, i in zip(last, idx):
            res[i] = torch.softmax(row[NUM[:len(jb[i][1])]], -1).cpu().numpy().round(4).tolist()
        if (s // B) % 200 == 0: print(f"  {s}/{len(jb)}  {time.time() - t:.0f}秒")
    json.dump({"n": len(jb), "head": jb[0][0][:200], "out": res}, open(ck, "w"))
    return res

STEP = 10000
GD = []
for c0 in range(0, len(ITEMS), STEP):
    part = ITEMS[c0:c0 + STEP]
    jb = jobs(part, HEADS, seed=c0, shuffle=True)
    GD += merge(jb, ask(jb, f"{OUT}/ck_gemma_{c0}.json"), 2)
    print("済み", len(GD), "/", len(ITEMS))
# 参考：WordNet の順のまま聞く（最初の意味が先頭に来る。E8 は辞書の順のまま聞いていた）
jb0 = jobs([ITEMS[i] for i in G0], HEADS, shuffle=False)
G0P = merge(jb0, ask(jb0, f"{OUT}/ck_gemma_wnorder.json"), 1)
del gm; torch.cuda.empty_cache()
print("Gemma 済み", len(GD), len(G0P))

## 3. 保存

In [ ]:
res = {"meta": {"gemma": GEMMA_ID, "emb": EMB_ID, "date": time.strftime("%Y-%m-%d"), "n": len(ITEMS),
                "data": "Princeton WordNet 3.0 gloss corpus (noun), tag=man, head by spaCy"},
       "ids": [x[0] for x in ITEMS], "sim": SIM, "gemma": GD, "g0_idx": G0, "gemma_wnorder": G0P}
path = f"{OUT}/E12_results.json.gz"
with gzip.open(path, "wt") as f:
    json.dump(res, f)
print("保存:", path, round(os.path.getsize(path) / 1e6, 2), "MB")